# Phase 1 ML Monitor and Executable Scanner Walkthrough

This notebook is the final walkthrough artifact for the prediction-markets Phase 1 system.

It covers:

1. the historical resolved-market data pipeline,
2. snapshot framing at open / mid / 24h,
3. offline model performance against market baselines,
4. current live monitor outputs,
5. executable scanner logic,
6. monitored detector runtime,
7. replay / feasibility analysis,
8. final conclusions and next steps.

This notebook is designed as a **review and explanation artifact**. It loads already-produced outputs from the repo rather than recomputing the full pipeline from scratch.

## Objective

Build a prediction-market research and monitoring system that:

1. learns from resolved historical markets,
2. benchmarks model performance against market-implied probabilities,
3. selects the strongest snapshot timing,
4. scores current live markets,
5. supports executable detector logic and replay-based feasibility checks.

## Pipeline overview

The repo now has four working layers:

1. **Historical forecasting / ML pipeline**
   - historical ingestion
   - resolved market datasets
   - snapshot generation
   - feature enrichment
   - offline evaluation

2. **Current-market scoring**
   - current Polymarket scoring
   - current Kalshi scoring
   - ranked outputs and confidence filtering

3. **Executable scanner logic**
   - executable price estimation from order books
   - fees and slippage-aware edge logic
   - complement and cross-venue detection
   - structured JSONL logging

4. **Replay / feasibility**
   - deterministic replay from logged detector inputs
   - latency-labelled replay scenarios
   - feasibility-style summary reporting

In [ ]:
from pathlib import Path
import json
import pandas as pd

BASE_DIR = Path.cwd()

paths = {
    "markets_recent": BASE_DIR / "data/processed/markets_recent.parquet",
    "open_recent_enriched": BASE_DIR / "data/processed/features_open_recent_enriched.parquet",
    "mid_recent_history_enriched": BASE_DIR / "data/processed/features_mid_recent_history_enriched.parquet",
    "t24_recent_history_enriched": BASE_DIR / "data/processed/features_24h_recent_history_enriched.parquet",
    "offline_summary_md": BASE_DIR / "reports/offline_snapshot_summary.md",
    "final_model_report": BASE_DIR / "reports/final_model_report.md",
    "replay_summary_md": BASE_DIR / "reports/replay_pred_summary.md",
    "current_monitor_report": BASE_DIR / "reports/current_monitor_report.md",
    "calibration_open": BASE_DIR / "reports/calibration_open_recent.png",
    "calibration_24h": BASE_DIR / "reports/calibration_recent_24h.png",
    "polymarket_scored": BASE_DIR / "data/processed/current_polymarket_scored_trained.parquet",
    "kalshi_scored": BASE_DIR / "data/processed/kalshi_current_scored_trained.parquet",
    "polymarket_features": BASE_DIR / "data/processed/current_polymarket_features.parquet",
    "kalshi_features": BASE_DIR / "data/processed/kalshi_current_features.parquet",
    "poly_top_edges_csv": BASE_DIR / "reports/current_polymarket_top_edges_trained.csv",
    "kalshi_top_edges_csv": BASE_DIR / "artifacts/outputs/kalshi_current_top_edges_trained.csv",
    "poly_runs_log": BASE_DIR / "logs/polymarket_complement_runs.jsonl",
    "poly_flags_log": BASE_DIR / "logs/polymarket_complement_flags.jsonl",
    "generic_flags_log": BASE_DIR / "logs/prediction_scanner_flags.jsonl",
}

pd.DataFrame(
    [{"artifact": k, "exists": p.exists(), "path": str(p)} for k, p in paths.items()]
)

## Dataset spine

The resolved-market dataset is the foundation of the offline monitor.

From this base dataset, the pipeline creates:
- an **open** snapshot dataset,
- a **midpoint** snapshot dataset,
- a **24h-before-close** snapshot dataset.

These are then enriched with market-implied probabilities and related features before model evaluation.

In [ ]:
markets_recent = pd.read_parquet(paths["markets_recent"])
open_recent = pd.read_parquet(paths["open_recent_enriched"])
mid_recent = pd.read_parquet(paths["mid_recent_history_enriched"])
t24_recent = pd.read_parquet(paths["t24_recent_history_enriched"])

summary_df = pd.DataFrame(
    [
        {"dataset": "markets_recent", "rows": len(markets_recent), "cols": len(markets_recent.columns)},
        {"dataset": "open_recent_enriched", "rows": len(open_recent), "cols": len(open_recent.columns)},
        {"dataset": "mid_recent_history_enriched", "rows": len(mid_recent), "cols": len(mid_recent.columns)},
        {"dataset": "24h_recent_history_enriched", "rows": len(t24_recent), "cols": len(t24_recent.columns)},
    ]
)

summary_df

## Snapshot logic

Each resolved market is evaluated at three points in time:

- **open (practical 12h near-open)**: first observed price within 12 hours of market creation
- **mid**: halfway between open and close
- **24h**: 24 hours before close

A strict open definition using the first observed price within 60 minutes of creation was implemented and tested, but the current history endpoint supports almost no usable rows under that rule. The working open evaluation therefore uses a practical near-open proxy.

## Open-window sensitivity

Open-window evaluation was tested under multiple definitions:

- **strict 60m**: effectively infeasible
- **practical 6h**: too sparse and worse than market
- **practical 12h**: best near-open compromise
- **practical 24h**: viable but looser

This supports using **12h** as the working near-open definition for Phase 1.

In [ ]:
open_window_comparison = pd.DataFrame(
    [
        {"open_window": "60m (strict)", "rows_used": 1, "model_brier": None, "market_brier": None, "takeaway": "infeasible"},
        {"open_window": "6h", "rows_used": 100, "model_brier": 0.135681, "market_brier": 0.094109, "takeaway": "market beats model"},
        {"open_window": "12h", "rows_used": 223, "model_brier": 0.117134, "market_brier": 0.129093, "takeaway": "model beats market"},
        {"open_window": "24h", "rows_used": 248, "model_brier": 0.136983, "market_brier": 0.150912, "takeaway": "model beats market"},
    ]
)

open_window_comparison

## Offline snapshot summary

The main offline comparison is across:

- open (practical 12h)
- mid
- 24h-before-close

The key question is whether the model beats:
- a naive 50/50 baseline, and
- the market-implied probability baseline.

In [ ]:
offline_summary_path = paths["offline_summary_md"]

if offline_summary_path.exists():
    print(offline_summary_path.read_text(encoding="utf-8"))
else:
    print("offline snapshot summary missing")

## Offline takeaway

The main result is:

- **24h** is the strongest overall modelling point
- **mid** is weaker than 24h but still useful
- **open (practical 12h)** is now viable and beats the market baseline
- **strict 60m open** is not supported at scale by the current history source

## Feature directionality

Coefficient inspection is used mainly for interpretation rather than as a production claim about causality.

The strongest and most stable feature direction is still the market-implied probability itself, with additional support from volume / liquidity style variables and some timing/context features depending on snapshot frame.

In [ ]:
coefficients_path = BASE_DIR / "reports/best_24h_coefficients.md"

if coefficients_path.exists():
    print(coefficients_path.read_text(encoding="utf-8"))
else:
    print("best_24h_coefficients.md missing")

## Current-market monitoring

The repo also supports scoring current live markets.

This is useful for:
- ranking current contracts,
- highlighting model-vs-market dislocations,
- comparing venue behavior,
- and feeding downstream scanner / detector logic.

## Live monitor and scanner outputs

This section shows the actual live-facing outputs of the project:

- current Polymarket scored monitor outputs,
- current Kalshi scored monitor outputs,
- current monitor report,
- recent scanner runtime logs,
- emitted flag logs,
- replay / feasibility summary.

The goal is to show the operational surface of the system, not just the offline modelling artifacts.

In [ ]:
poly_scored_path = paths["polymarket_scored"]

if poly_scored_path.exists():
    poly_scored_df = pd.read_parquet(poly_scored_path)
    print("Polymarket scored rows:", len(poly_scored_df))
    display(poly_scored_df.head(20))
else:
    print("current_polymarket_scored_trained.parquet missing")

In [ ]:
kalshi_scored_path = paths["kalshi_scored"]

if kalshi_scored_path.exists():
    kalshi_scored_df = pd.read_parquet(kalshi_scored_path)
    print("Kalshi scored rows:", len(kalshi_scored_df))
    display(kalshi_scored_df.head(20))
else:
    print("kalshi_current_scored_trained.parquet missing")

### Ranked current monitor outputs

These CSV exports are useful for showing the highest-ranked current opportunities or model-vs-market dislocations.

In [ ]:
poly_edges_csv = paths["poly_top_edges_csv"]

if poly_edges_csv.exists():
    poly_edges_df = pd.read_csv(poly_edges_csv)
    print("Polymarket top edges rows:", len(poly_edges_df))
    display(poly_edges_df.head(20))
else:
    print("current_polymarket_top_edges_trained.csv missing")

In [ ]:
kalshi_edges_csv = paths["kalshi_top_edges_csv"]

if kalshi_edges_csv.exists():
    kalshi_edges_df = pd.read_csv(kalshi_edges_csv)
    print("Kalshi top edges rows:", len(kalshi_edges_df))
    display(kalshi_edges_df.head(20))
else:
    print("kalshi_current_top_edges_trained.csv missing")

In [ ]:
monitor_report_path = paths["current_monitor_report"]

if monitor_report_path.exists():
    print(monitor_report_path.read_text(encoding="utf-8"))
else:
    print("current_monitor_report.md missing")

## Executable scanner logic

Beyond offline modelling, the repo now includes an executable detector stack that:

- walks order books to estimate executable buy / sell prices,
- applies fee and slippage-aware costs,
- evaluates complement and cross-venue opportunities,
- writes structured JSONL logs for flags and monitored runs.

This extends the system beyond offline forecasting into executable monitoring and validation.

## Detector runtime evidence

The live detector runtime was monitored over extended runs and remained stable across repeated cycles.

The monitored runs demonstrate:
- repeated successful orderbook ingestion,
- stable paired-market coverage,
- sufficient executable-size coverage,
- structured run logging for later validation.

This establishes the detector as operational, even where emitted live opportunities remain dependent on live market conditions and thresholds.

In [ ]:
poly_run_log = paths["poly_runs_log"]

if poly_run_log.exists():
    poly_run_rows = [json.loads(line) for line in poly_run_log.read_text(encoding="utf-8").splitlines() if line.strip()]
    poly_run_df = pd.DataFrame(poly_run_rows)

    print("Polymarket complement run rows:", len(poly_run_df))
    display(poly_run_df.tail(20))

    for col in [
        "rows_total",
        "paired_markets",
        "markets_with_both_asks",
        "sufficient_size",
        "flags_emitted",
        "duration_seconds",
    ]:
        if col in poly_run_df.columns:
            print(col, "avg =", poly_run_df[col].mean())
else:
    print("polymarket_complement_runs.jsonl missing")

### Live emitted flags: Polymarket complement loop

If live executable complement flags were emitted, they will appear here.

In [ ]:
poly_flag_log = paths["poly_flags_log"]

if poly_flag_log.exists():
    poly_flag_rows = [json.loads(line) for line in poly_flag_log.read_text(encoding="utf-8").splitlines() if line.strip()]
    poly_flag_df = pd.DataFrame(poly_flag_rows)

    if len(poly_flag_df) > 0:
        print("Polymarket complement emitted flags:", len(poly_flag_df))
        display(poly_flag_df.tail(20))
    else:
        print("polymarket_complement_flags.jsonl exists but contains no rows")
else:
    print("polymarket_complement_flags.jsonl missing")

### Generic scanner flags

This log is useful because it includes the richer structured flag schema used in synthetic and replayable detector examples.

In [ ]:
generic_flag_log = paths["generic_flags_log"]

if generic_flag_log.exists():
    generic_flag_rows = [json.loads(line) for line in generic_flag_log.read_text(encoding="utf-8").splitlines() if line.strip()]
    generic_flag_df = pd.DataFrame(generic_flag_rows)

    print("Generic prediction scanner flags:", len(generic_flag_df))
    display(generic_flag_df.tail(20))
else:
    print("prediction_scanner_flags.jsonl missing")

## Replay / feasibility

Replay was implemented in two stages:

- deterministic replay from logged detector inputs
- latency-labelled replay scenarios to estimate feasibility under delayed execution

This allows replay summaries to estimate whether logged opportunities remain positive under latency assumptions, and provides the basis for later observed-state replay.

In [ ]:
replay_summary_path = paths["replay_summary_md"]

if replay_summary_path.exists():
    print(replay_summary_path.read_text(encoding="utf-8"))
else:
    print("replay_pred_summary.md missing")

### Live-layer interpretation

The live layer now has three distinct functions:

1. **Current monitor**
   - scores live Polymarket and Kalshi markets

2. **Executable scanner**
   - evaluates executable opportunities using live order books and structured detection logic

3. **Replay / feasibility**
   - estimates whether detected opportunities remain positive under latency assumptions

This means the repo now supports not just offline modelling, but also a real operational monitor / scanner / replay workflow.

In [ ]:
summary_rows = []

if paths["polymarket_scored"].exists():
    summary_rows.append({"component": "Polymarket scored monitor", "rows": len(pd.read_parquet(paths["polymarket_scored"]))})

if paths["kalshi_scored"].exists():
    summary_rows.append({"component": "Kalshi scored monitor", "rows": len(pd.read_parquet(paths["kalshi_scored"]))})

if paths["poly_runs_log"].exists():
    poly_run_rows = [json.loads(line) for line in paths["poly_runs_log"].read_text(encoding="utf-8").splitlines() if line.strip()]
    summary_rows.append({"component": "Polymarket complement run log", "rows": len(poly_run_rows)})

if paths["poly_flags_log"].exists():
    poly_flag_rows = [json.loads(line) for line in paths["poly_flags_log"].read_text(encoding="utf-8").splitlines() if line.strip()]
    summary_rows.append({"component": "Polymarket complement flags", "rows": len(poly_flag_rows)})

if paths["generic_flags_log"].exists():
    generic_flag_rows = [json.loads(line) for line in paths["generic_flags_log"].read_text(encoding="utf-8").splitlines() if line.strip()]
    summary_rows.append({"component": "Generic replayable scanner flags", "rows": len(generic_flag_rows)})

pd.DataFrame(summary_rows)

## Final artifacts produced

The main delivery artifacts now include:

- resolved-market parquet datasets
- open / mid / 24h enriched snapshot datasets
- calibration plots
- offline snapshot summary
- current monitor report
- final model report
- detector runtime logs
- replay / feasibility summary

In [ ]:
final_artifacts = [
    "data/processed/features_open_recent_enriched.parquet",
    "data/processed/features_mid_recent_history_enriched.parquet",
    "data/processed/features_24h_recent_history_enriched.parquet",
    "reports/offline_snapshot_summary.md",
    "reports/current_monitor_report.md",
    "reports/final_model_report.md",
    "reports/replay_pred_summary.md",
    "reports/calibration_open_recent.png",
    "reports/calibration_recent_24h.png",
]

artifact_df = pd.DataFrame(
    [{"artifact": p, "exists": (BASE_DIR / p).exists()} for p in final_artifacts]
)

artifact_df

## Conclusions

### Main modelling conclusion
The **24h snapshot** is the strongest overall modelling point.

### Open-window conclusion
A strict 60-minute open definition is not viable with the current history source. A **practical 12h near-open** definition is the best current compromise.

### Live-monitor conclusion
- **Polymarket** is the cleaner and more trustworthy live scorer.
- **Kalshi** is integrated and useful, but still more exploratory due to cross-venue domain shift.

### Scanner conclusion
The repo now supports a functional end-to-end scanner pipeline with executable pricing, fee-aware edge logic, structured logging, and replay / feasibility analysis.

### Most important project outcome
The system is no longer just an offline modelling exercise. It now supports a real current-market monitoring and detector-validation workflow.

## What I would tell Mario

The Phase 1 ML monitor is now complete in a delivery-ready form. The historical ingestion, leakage-safe snapshot framing, enriched parquet outputs, and offline evaluation across open, mid, and 24h windows are all working, and the model beats the market baseline in the strongest offline settings. For open, a strict 60-minute definition was implemented and tested, but the current history source supports almost no usable rows under that rule, so the working open evaluation now uses a tighter practical 12-hour near-open definition. In parallel, the repo now includes executable detector logic with book-walking, fee/slippage-aware edge computation, structured JSONL logging, monitored runtime, and replay / feasibility analysis. The scanner is functional end-to-end, and the main remaining work is continuing to strengthen live opportunity-generation evidence under current thresholds and data conditions, while hardening observed-state replay.

## Recommended next steps

1. strengthen live opportunity generation evidence under current or adjusted thresholds,
2. continue observed-state replay development,
3. refine dashboard / walkthrough presentation,
4. continue hardening calibration and detector validation as more live data accumulates.